# Lab 14 · Frame the debate and agree on a message contract

**Prepared by Mustafa Assaf** · Self-study: Build your council further · 15–20 minute lab

## Before the debate: a frame and a contract

**So far:** in Lab 06, the architects answered a fixed question with free-form claims, and the chair checked only their sources.

**New:** two things a council needs before it debates.

- **A frame.** The chair turns the brief into *quality-attribute scenarios*. Each names a quality characteristic from the ISO/IEC 25010 quality model, a *stimulus* (what happens), a *response* (what the system must do), and a *measure* (a number that shows whether it did). Every architect then argues about the same scenarios.
- **A message contract.** Every proposal must name an option, mark a trade-off for each scenario, give a confidence from 0 to 1, and state a claim. The chair rejects anything else. Because the code depends only on the contract, adding an architect takes one line.

The case is new: a Booking service, not the Payments platform.

**At a glance**

![Lab 14 at a glance: Brief (Booking service), then Chair frames (3 ISO 25010 scenarios), then has numbers?, then 4 architects (one contract), then Contract check (option · marks · confidence), then Aggregate (votes vs confidence).](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/14.svg)

## Pattern · Passive Goal Creator

*Single-agent pattern from the [Architectural Patterns for Agentic AI](https://karthikv1392.github.io/agentic-patterns/) field guide. Related: Registry & Adapter.*

| | |
| --- | --- |
| **Context** | People state broad goals rather than precise instructions, so an agent must first give the goal structure. |
| **Problem** | Sent directly, an ambiguous goal leaves the hidden sub-tasks unstated, and the answer becomes shallow. |
| **Solution** | Decompose the goal into explicit sub-tasks first, then solve them and combine the results. |
| **Use when** | The task is open-ended; a transparent first step is useful; no external context is needed. |
| **Trade-off** | Simple and easy to inspect, but a wrong decomposition propagates through every sub-task. |
| **Flow** | Goal → Decompose → Solve → Answer |
| **Quality attributes** | Benefits: Predictability, Auditability, Maintainability, Usability. Liabilities: Reliability, Accuracy, Adaptability. |
| **In this lab** | The chair decomposes the brief into quality-attribute scenarios before anyone proposes. The message contract then plays the role of *Registry & Adapter*'s uniform interface, so that architects can be added or replaced without changing the council's code (the guide's Modifiability benefit). |

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Practice answers are scripted. For a live model, choose `PROVIDER` (`"freeinference"`, `"gemini"`, or `"deepseek"`), save its key in Colab Secrets (`FREEINFERENCE_API_KEY`, `GEMINI_API_KEY`, or `DEEPSEEK_API_KEY`), and change `MODE` to `"live"`. `ask` stops after 12 calls; run this cell again to reset the count.

In [ ]:
from importlib.util import find_spec
if not all(find_spec(name) for name in ['coursekit']):
    %pip install -q "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/9c78f2181b0f793c60b2b081c79ae74276026e7d.zip"
from coursekit import make_ask
MODE = "practice"
PROVIDER = "freeinference"  # or "gemini" or "deepseek"
ask = make_ask(mode=MODE, provider=PROVIDER)

## Step 1 · A shared vocabulary for quality

`QUALITIES` lists the nine product quality characteristics of ISO/IEC 25010:2023. A fixed vocabulary lets code check the frame, and lets two architects mean the same thing by "security". `scenario_problems` is the gate for one scenario; the last line tries it on a vague one.

In [ ]:
import json

QUALITIES = ["functional suitability", "performance efficiency", "compatibility", "interaction capability",
             "reliability", "security", "maintainability", "flexibility", "safety"]  # ISO/IEC 25010:2023
brief = ("Booking service: 8 developers in 2 squads. Bookings spike to 50 per second on Monday mornings. "
         "A payment partner requires an audit log of every change. One engineer runs operations.")

def scenario_problems(s):
    """A scenario names an ISO 25010 quality, a stimulus, a response, and a measure with a number."""
    problems = [f"missing {key}" for key in ("quality", "stimulus", "response", "measure") if not s.get(key)]
    if str(s.get("quality", "")).lower() not in QUALITIES:
        problems.append(f"unknown quality {s.get('quality')!r}")
    if not any(ch.isdigit() for ch in str(s.get("measure", ""))):
        problems.append("the measure has no number")
    return problems

print(scenario_problems({"quality": "speed", "measure": "fast"}))

## Step 2 · The chair frames the case

One model call turns the brief into three scenarios, and the gate checks each of them. A frame without numbers cannot be checked later, so the gate insists on a measure.

In [ ]:
scripted_frame = json.dumps([
    {"quality": "performance efficiency", "stimulus": "50 bookings per second on Monday morning",
     "response": "every booking is confirmed", "measure": "95% within 1 second"},
    {"quality": "security", "stimulus": "an auditor asks who changed a booking",
     "response": "the audit log names the change and its author", "measure": "100% of changes logged"},
    {"quality": "maintainability", "stimulus": "a squad changes its own feature",
     "response": "it releases without waiting for the other squad", "measure": "release within 1 day"}])
prompt = (f"Brief: {brief}\nFrame the architecture decision as 3 quality-attribute scenarios. "
          f"Use only these ISO/IEC 25010 characteristics: {QUALITIES}. Reply with JSON only, no code fences: "
          'a list of objects with "quality", "stimulus", "response", and "measure" (a target with a number).')
scenarios = json.loads(ask(prompt, example=scripted_frame))
for s in scenarios:
    print(f"{s['quality']:<23} {s['measure']:<22} problems: {scenario_problems(s) or 'none'}")
assert not any(scenario_problems(s) for s in scenarios), "The frame breaks the scenario rules."

## Step 3 · The message contract

`contract_problems` lists everything wrong with one proposal, as Lab 01's parser and Lab 03's gates did. Each rule is one line in `rules`, so a new rule is easy to add. The contract knows the frame, so it can demand a `"+"` or `"-"` for each scenario. The `bool` check is there because Python treats `True` as the number 1. `FORMAT` tells the architects the same contract in words.

In [ ]:
OPTIONS = ["modular monolith", "microservices"]
FORMAT = ('Reply with JSON only, no code fences: {"option": ..., "confidence": 0 to 1, '
          '"tradeoffs": {quality: "+" or "-"}, "claim": "one sentence"}')

def contract_problems(msg):
    """Everything a proposal must carry before the chair accepts it. An empty list means it passes."""
    marks, confidence = msg.get("tradeoffs", {}), msg.get("confidence")
    unmarked = [s["quality"] for s in scenarios if marks.get(s["quality"]) not in ("+", "-")]
    rules = {f"option must be one of {OPTIONS}": msg.get("option") in OPTIONS,
             f"no trade-off mark for {unmarked}": not unmarked,
             "confidence must be a number from 0 to 1": isinstance(confidence, (int, float))
             and not isinstance(confidence, bool) and 0 <= confidence <= 1,
             "a one-sentence claim is required": bool(msg.get("claim"))}
    return [problem for problem, ok in rules.items() if not ok]

print(contract_problems({"option": "serverless", "confidence": "high"}))

## Step 4 · Four architects, one contract

The council now has four architects. The fourth one, the Auditor, took one line in `architects` and one scripted reply: nothing else changed. Each reply is checked against the contract, and a reply that breaks it is rejected with its problems. In practice mode, the Auditor's scripted confidence is the word `"high"`.

In [ ]:
architects = {  # add an architect with one more line: the code below depends only on the contract
    "Minimalist": "Prefer the simplest design that meets the scenarios.",
    "Safeguard": "Prefer low-risk designs; weigh operations and audits.",
    "Modernizer": "Prefer designs that let each squad release on its own.",
    "Auditor": "Demand a measurable reason for every claim.",
}
scripted = {  # practice replies: (option, confidence, marks for the three scenarios, in order)
    "Minimalist": ("modular monolith", 0.7, "++-"), "Safeguard": ("modular monolith", 0.6, "++-"),
    "Modernizer": ("microservices", 0.9, "+-+"), "Auditor": ("modular monolith", "high", "++-")}
accepted = {}
for name, stance in architects.items():
    option, confidence, signs = scripted[name]
    example = json.dumps({"option": option, "confidence": confidence, "claim": f"The {name}'s view.",
                          "tradeoffs": dict(zip([s["quality"] for s in scenarios], signs))})
    task = (f"You are the {name}. {stance}\nBrief: {brief}\n"
            f"Scenarios: {json.dumps(scenarios)}\nChoose one of {OPTIONS}. {FORMAT}")
    reply = json.loads(ask(task, example=example))
    problems = contract_problems(reply)
    print(f"{name:<11}", f"REJECTED: {problems}" if problems else f"{reply['option']}, confidence {reply['confidence']}")
    accepted.update({} if problems else {name: reply})

## Step 5 · Votes and confidence

Two ways to aggregate the accepted proposals: one vote each, or the sum of their confidence. Here they agree. But a confidence is the model's own claim about itself, and models are often overconfident. Record it, and do not let it outweigh the votes until you have checked it against outcomes, as Lab 15 does with each architect's record.

In [ ]:
votes, confidence = {}, {}
for proposal in accepted.values():
    votes[proposal["option"]] = votes.get(proposal["option"], 0) + 1
    confidence[proposal["option"]] = round(confidence.get(proposal["option"], 0) + proposal["confidence"], 2)
print("votes:", votes, "| confidence:", confidence)

## What you should see

Step 1 prints four problems for the vague scenario: no stimulus, no response, an unknown quality, and a measure without a number. Step 2 prints three scenarios with no problems. Step 3 prints four problems for the bad proposal. In Step 4, three architects are accepted and the Auditor is rejected: `confidence must be a number from 0 to 1`. Step 5 prints `votes: {'modular monolith': 2, 'microservices': 1} | confidence: {'modular monolith': 1.3, 'microservices': 0.9}`.

## Try it

1. Fix the Auditor's confidence to `0.4`. Do the votes or the confidence sums change the decision?
2. Make the Modernizer's confidence `1.0` and the Minimalist's `0.2`. Votes and confidence now disagree. Which would you trust, and why?
3. Add a fifth architect with your own stance: one line in `architects` and one scripted reply.
4. Change the scripted frame so that one measure reads `"quickly"`. Which step stops, and why is that the right place?
5. **Use it in your council.** Write the frame and the contract for your project's first scenario. Which quality characteristics does it need, and what number measures each one?

## Takeaway

A frame gives every architect the same scenarios to argue about, in a shared vocabulary, with numbers that can be checked. A message contract makes every proposal comparable and every agent replaceable: the council's code depends on the contract, not on who sent the message. Self-reported confidence is information, not proof.

## Read further

- [ISO/IEC 25010:2023 product quality model](https://www.iso.org/standard/78176.html)
- [Bass, Clements, and Kazman: quality attribute scenarios (SEI)](https://insights.sei.cmu.edu/library/software-architecture-in-practice-fourth-edition/)